In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install highway-env
%pip install stable-baselines3
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 14.3 MB/s eta 0:00:00


In [3]:
%cd /content/drive/MyDrive/cs-272-final-project/highway/best-model

/content/drive/MyDrive/cs-272-final-project/highway/best-model


In [4]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning,)
warnings.filterwarnings("ignore", message=".*Kernel._parent_header.*")

In [5]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from optuna.exceptions import TrialPruned
import gymnasium as gym
import highway_env  # registers env ids
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ------------------------------ Config ------------------------------
SEED = 42
ENV_ID = "highway-fast-v0"


# Folder layout (run this from highway/hyperparam-tuned or any folder you want)
BASE_DIR     = os.getcwd()
RUNS_ROOT    = os.path.join(BASE_DIR, "runs_monitor_highway")     # per-trial monitor logs
FINAL_ROOT   = os.path.join(RUNS_ROOT, "final_best")      # final training logs
MODELS_DIR   = os.path.join(BASE_DIR, "models")
PLOTS_DIR    = os.path.join(BASE_DIR, "plots")
TB_DIR      = os.path.join(BASE_DIR, "runs_tensorboard")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ------------------------------ Helpers ------------------------------

def make_train_env(monitor_dir, n_envs=4):
    gray_config = {
    "observation": {
        "type": "GrayscaleObservation",
        "observation_shape": (128, 64),
        "stack_size": 4,
        "weights": [0.2989, 0.5870, 0.1140],  # weights for RGB conversion
        "scaling": 1.75,
      },
    }

    os.makedirs(monitor_dir, exist_ok=True)
    return make_vec_env(
        ENV_ID,
        n_envs=n_envs,
        seed=SEED,
        env_kwargs={"config": gray_config},
        monitor_dir=monitor_dir,   # logs training episodes to CSVs
    )

# --- Plot 1: learning curve from Monitor CSVs (episodes on X) ---
def plot_learning_curve_monitor(monitor_dir: str, out_png: str, smooth_window: int = 50, title="Learning Curve — Highway(Grayscale) - PPO(tuned)"):
    files = sorted(glob.glob(os.path.join(monitor_dir, "**", "*.csv"), recursive=True))
    if not files:
        raise FileNotFoundError(f"No monitor CSVs under: {monitor_dir}")

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, comment="#")
            if "r" in df.columns:
                dfs.append(df[["r"]].copy())
        except Exception:
            pass
    if not dfs:
        raise RuntimeError("Found CSVs but no episodic returns (column 'r')")

    log = pd.concat(dfs, ignore_index=True)
    log["episode"] = np.arange(1, len(log) + 1)
    win = max(1, min(smooth_window, len(log)))
    log["r_smooth"] = log["r"].rolling(win, min_periods=1).mean()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.plot(log["episode"], log["r_smooth"])
    plt.xlabel("Training Episodes")
    plt.ylabel("Mean Episodic Return (smoothed)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"[plot] saved: {out_png}")

# --- Plot 2: 500-episode deterministic evaluation violin ---
def evaluate_and_plot_violin(model_path: str, out_png: str,
                             env_id: str = ENV_ID,
                             n_episodes: int = 500,
                             deterministic: bool = True,
                             seed: int = SEED):
    env = gym.make(env_id)
    model = PPO.load(model_path)

    returns = []
    for _ in range(n_episodes):
        obs, info = env.reset(seed=seed)
        done = False
        ep_ret = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)  # no exploration
            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += reward
            done = terminated or truncated
        returns.append(ep_ret)
    env.close()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.violinplot([returns], showmeans=True)
    plt.xticks([1], ["PPO"])
    plt.ylabel(f"Episodic Return (n={n_episodes}, deterministic)")
    plt.title("Performance Test — Highway(Grayscale) Env PPO (tuned)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"[plot] saved: {out_png}")

# --- Metric from Monitor CSVs: mean of last-K episode returns ---
def last_k_mean(monitor_dir: str, k: int = 200) -> float:
    files = sorted(glob.glob(os.path.join(monitor_dir, "**", "*.csv"), recursive=True))
    vals = []
    for f in files:
        try:
            df = pd.read_csv(f, comment="#")
            if "r" in df.columns:
                vals.extend(df["r"].tolist())
        except Exception:
            pass
    if not vals:
        return -1e9
    vals = vals[-k:] if len(vals) >= k else vals
    return float(np.mean(vals))

In [ ]:
# ------------------------------ Optuna objective ------------------------------
def objective(trial: optuna.trial.Trial) -> float:
    # Search space (solid defaults for PPO)
    lr         = trial.suggest_float("lr", 1e-5, 3e-3, log=True)
    n_steps    = trial.suggest_categorical("n_steps", [1024, 2048])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    clip       = trial.suggest_float("clip_range", 0.1, 0.3)
    gae_lam    = trial.suggest_float("gae_lambda", 0.90, 0.98)
    gamma      = trial.suggest_float("gamma", 0.97, 0.999)
    ent_coef   = trial.suggest_float("ent_coef", 1e-4, 1e-2, log=True)
    n_epochs   = trial.suggest_categorical("n_epochs", [5, 10])

    trial_dir = os.path.join(RUNS_ROOT, f"trial_{trial.number:03d}")
    env = make_train_env(trial_dir, n_envs=4)

    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,
        seed=SEED,
        learning_rate=lr,
        n_steps=n_steps,
        batch_size=batch_size,
        clip_range=clip,
        gae_lambda=gae_lam,
        gamma=gamma,
        ent_coef=ent_coef,
        n_epochs=n_epochs,
    )

    # ---- run fixed number of PPO updates; prune early if underperforming ----
    n_updates = 8                                # small, fast trials
    steps_per_update = n_steps * 4               # n_envs = 4
    for u in range(n_updates):
        model.learn(total_timesteps=steps_per_update,
                    reset_num_timesteps=False, progress_bar=False)

        if u >= 3:  # warmup a few updates before judging
            score = last_k_mean(trial_dir, k=100)
            trial.report(score, step=u)
            if trial.should_prune():
                env.close()
                raise TrialPruned()

    env.close()

    # Score = mean last-K ep returns from Monitor CSVs
    return last_k_mean(trial_dir, k=200)

In [ ]:
def main():
    os.makedirs(RUNS_ROOT, exist_ok=True)

    # 1) Hyperparameter search
    sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=10, timeout=11*60*60)

    print("Best trial:", study.best_trial.number)
    print("Best value (last-k mean):", study.best_value)
    print("Best params:", study.best_params)

    with open(os.path.join(MODELS_DIR, "ppo_best_params_highway_grayscale.json"), "w") as f:
        json.dump(study.best_params, f, indent=2)

    # 2) Final training with best params (longer budget)
    os.makedirs(FINAL_ROOT, exist_ok=True)
    env = make_train_env(FINAL_ROOT, n_envs=4)

    bp = study.best_params
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        seed=SEED,
        learning_rate=bp["lr"],
        n_steps=bp["n_steps"],
        batch_size=bp["batch_size"],
        clip_range=bp["clip_range"],
        gae_lambda=bp["gae_lambda"],
        gamma=bp["gamma"],
        ent_coef=bp["ent_coef"],
        n_epochs=bp["n_epochs"],
        tensorboard_log=TB_DIR,
    )

    total_timesteps = 204800 # Make it comparabale to baseline
    model.learn(total_timesteps=total_timesteps, progress_bar=False, tb_log_name="Highway_grayscale")
    env.close()

    final_model_path = os.path.join(MODELS_DIR, "ppo_highway_grayscale_env_tuned")
    model.save(final_model_path)
    print(f"[model] saved: {final_model_path}.zip")

    # 3) Plots
    plot_learning_curve_monitor(
        FINAL_ROOT,
        os.path.join(PLOTS_DIR, "learning_highway_grayscale_env_tuned.png"),
    )

    evaluate_and_plot_violin(
        final_model_path,
        os.path.join(PLOTS_DIR, "violin_highway_grayscale_env_tuned.png"),
        env_id=ENV_ID,
        n_episodes=500,
        deterministic=True,
    )

if __name__ == "__main__":
    main()

[I 2025-12-07 22:41:17,070] A new study created in memory with name: no-name-71c18498-c9d1-42fd-8674-0e32d4a88115
[I 2025-12-07 23:13:36,726] Trial 0 finished with value: 14.328973785 and parameters: {'lr': 8.468008575248323e-05, 'n_steps': 1024, 'batch_size': 64, 'clip_range': 0.1116167224336399, 'gae_lambda': 0.9692940916619948, 'gamma': 0.9874323353405531, 'ent_coef': 0.0026070247583707684, 'n_epochs': 10}. Best is trial 0 with value: 14.328973785.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[I 2025-12-07 23:42:44,518] Trial 1 finished with value: 16.7729738 and parameters: {'lr': 0.0011536162338241388, 'n_steps': 1024, 'batch_size': 256, 'clip_range': 0.18638900372842315, 'gae_lambda': 0.9232983312158434, 'gam

Best trial: 6
Best value (last-k mean): 19.154632215
Best params: {'lr': 0.002460593946001259, 'n_steps': 2048, 'batch_size': 128, 'clip_range': 0.2891619439947426, 'gae_lambda': 0.9465551559096791, 'gamma': 0.9987641164516213, 'ent_coef': 0.00011545237112193649, 'n_epochs': 5}
Using cpu device
Logging to /content/drive/MyDrive/cs-272-final-project/highway/best-model/runs_tensorboard/Highway_grayscale_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 9.84     |
|    ep_rew_mean     | 7.46     |
| time/              |          |
|    fps             | 22       |
|    iterations      | 1        |
|    time_elapsed    | 362      |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 15.3        |
|    ep_rew_mean          | 11.2        |
| time/                   |             |
|    fps                  | 21          |
|

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[plot] saved: /content/drive/MyDrive/cs-272-final-project/highway/best-model/plots/learning_highway_grayscale_env_tuned.png


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ValueError: axes don't match array

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# Running this due to some error

def evaluate_and_plot_violin(model_path: str, out_png: str,
                             env_id: str = "highway-fast-v0",
                             n_episodes: int = 500,
                             deterministic: bool = True,
                             seed: int = 42):

    gray_config = {
    "observation": {
        "type": "GrayscaleObservation",
        "observation_shape": (128, 64),
        "stack_size": 4,
        "weights": [0.2989, 0.5870, 0.1140],  # weights for RGB conversion
        "scaling": 1.75,
      },
    }

    env = gym.make(env_id, config=gray_config)   # <-- pass the config!
    model = PPO.load(model_path)

    returns = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        done = False
        ep_ret = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += reward
            done = terminated or truncated
        returns.append(ep_ret)
    env.close()

    plt.figure()
    plt.violinplot([returns], showmeans=True)
    plt.xticks([1], ["PPO"])
    plt.ylabel(f"Episodic Return (n={n_episodes}, deterministic)")
    plt.title("Performance Test — Highway(Grayscale) Env PPO (tuned)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()

def main():
    BASE_DIR     = os.getcwd()
    MODELS_DIR   = os.path.join(BASE_DIR, "models")
    PLOTS_DIR    = os.path.join(BASE_DIR, "plots")
    MODEL_ZIP = os.path.join(MODELS_DIR, "ppo_highway_grayscale_env_tuned.zip")

    evaluate_and_plot_violin(MODEL_ZIP, os.path.join(PLOTS_DIR, "violin_highway_grayscale_env_tuned.png"))

if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
